# Modifying SEOBNRv5HMROM with pSEOB ringdown deviations

This notebook builds frequency-domain injections by using `SEOBNRv5HMROM` as the carrier waveform and transferring the mode-by-mode amplitude and phase changes obtained from deviated `pSEOBNRv5HM` waveforms. For each mode, the construction is

$$
\tilde h_{\ell m}^{\mathrm{modified}}(f)
=\tilde h_{\ell m}^{\mathrm{ROM}}(f)
\frac{A_{\ell m}^{\mathrm{non-GR}}(f)}{A_{\ell m}^{\mathrm{GR}}(f)}
\exp\!\left\{i\left[\phi_{\ell m}^{\mathrm{non-GR}}(f)-\phi_{\ell m}^{\mathrm{GR}}(f)\right]\right\}.
$$

The deviation scan uses only $M=10^6\,M_\odot$ and $q=4$. Both notebooks use the same scan tag and data directory. The 18 scan points keep $d_\omega=d_\tau$, including the zero-deviation control and the existing 0.02 point. Original injection IDs 0–6 are retained; new values are appended. The grid adds 0.025 and 0.075 beside the existing 0.05, then 0.0025/0.005/0.0075, 0.00025/0.0005/0.00075, and 0.000025/0.00005/0.000075 in the final interval down to zero. The deviation is introduced in the $(2,2)$ mode, while all requested modes are retained when constructing the LISA $A$ and $E$ channels.


In [1]:
import copy
import json
import os
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

import astropy.units as u
from astropy.cosmology import Planck15 as cosmo
from pyseobnr.generate_waveform import GenerateWaveform


/pbs/home/a/amahdi/.conda/envs/lisabeta/lib/python3.10/site-packages/pyseobnr/generate_waveform.py:6: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [2]:
import lisabeta.pyconstants as pyconstants
import lisabeta.tools.pytools as pytools
import lisabeta.lisa.pyresponse as pyresponse
import lisabeta.lisa.pyLISAnoise as pyLISAnoise
import lisabeta.lisa.lisatools as lisatools
import lisabeta.lisa.lisa as lisa


In [3]:
%matplotlib inline


## 1. Plot style

Here we set a plotting style so that the diagnostic figures remain consistent throughout the notebook.


In [4]:
plot_style = {
    "font.family": "serif",
    "axes.unicode_minus": False,
    "text.usetex": False,
    "axes.grid": True,
    "grid.linestyle": ":",
    "grid.linewidth": 1.0,
    "hist.bins": 50,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,
    "savefig.bbox": "tight",
    "savefig.transparent": True,
    "figure.dpi": 100,
}

plt.rcParams.update(plot_style)


## 2. Directories

Defining directories for saving both the input and output.


In [5]:
Syst_dir = os.path.expanduser("/pbs/home/a/amahdi/pySEOBNR_deviation_2/Data")
os.makedirs(Syst_dir, exist_ok=True)

# Using a common tag for the complete single-mass scan
deviation_tag = "domega_dtau_scan_M1e6_q4_dense"

Input_scan_dir = os.path.join(Syst_dir, "pSEOB_" + deviation_tag)

Output_root = os.path.join(
    Syst_dir,
    "SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_" + deviation_tag,
)
os.makedirs(Output_root, exist_ok=True)

print("Syst_dir:", Syst_dir)
print("Input_scan_dir:", Input_scan_dir)
print("Writing new files under Output_root:", Output_root)


Syst_dir: /pbs/home/a/amahdi/pySEOBNR_deviation_2/Data
Input_scan_dir: /pbs/home/a/amahdi/pySEOBNR_deviation_2/Data/pSEOB_domega_dtau_scan_M1e6_q4_dense
Writing new files under Output_root: /pbs/home/a/amahdi/pySEOBNR_deviation_2/Data/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e6_q4_dense


## 3. Waveform configuration

Defining the modes, deviation values, and common waveform settings before starting the scan through the deviation values and total binary masses (in solar mass).


In [6]:
common_modes = [(2, 2), (2, 1), (3, 3), (4, 4), (4, 3), (5, 5)]

# One total mass and its folder label
MASS_VALUES = np.array([1e6], dtype=float)
MASS_LABELS = ["M1e6"]

if len(MASS_VALUES) != len(MASS_LABELS):
    raise ValueError("MASS_VALUES and MASS_LABELS must have the same length.")

# Setting the dOmega/dTau values
# Original IDs 0--6 are preserved. New points fill each interval, including 0 to 1e-4.
deviation_values = np.array(
    [0.01, 0.001, 0.0001, 0.0, 0.0025, 0.005, 0.0075,
     0.00025, 0.0005, 0.00075, 0.000025, 0.00005, 0.000075],
    dtype=float,
)
N = len(deviation_values)
MASS_RATIO = 4.0
ZERO_DEVIATION_ID = int(np.flatnonzero(deviation_values == 0.0)[0])

deviation_modes = [(2, 2)]

OVERWRITE_OUTPUTS = True

# Using the injection ID and mass label to distinguish individual output files
output_tag = "rom_pseob_ratio_phase_" + deviation_tag

# Keeping the first scan point as a default before entering the injection loop
d_omega = float(deviation_values[0])
d_tau = float(deviation_values[0])


def make_deviation_dicts(d_omega_value, d_tau_value, modes=None):
    """Building pySEOBNR dOmega/dTau dictionaries for one scan row."""
    if modes is None:
        modes = common_modes

    domega_this = {
        f"{l},{m}": (float(d_omega_value) if (l, m) in deviation_modes else 0.0)
        for (l, m) in modes
    }
    dtau_this = {
        f"{l},{m}": (float(d_tau_value) if (l, m) in deviation_modes else 0.0)
        for (l, m) in modes
    }
    return domega_this, dtau_this


domega_dict, dtau_dict = make_deviation_dicts(d_omega, d_tau, modes=common_modes)

print("Modes:", common_modes)
print("Mass values:", MASS_VALUES)
print("Mass folders:", MASS_LABELS)
print("N scan points per mass:", N)
print("Deviation values:", deviation_values)
print("Deviation tag:", deviation_tag)
print("Default domega_dict:", domega_dict)
print("Default dtau_dict:", dtau_dict)


Modes: [(2, 2), (2, 1), (3, 3), (4, 4), (4, 3), (5, 5)]
Mass values: [1000000.]
Mass folders: ['M1e6']
N scan points per mass: 13
Deviation values: [1.0e-02 1.0e-03 1.0e-04 0.0e+00 2.5e-03 5.0e-03 7.5e-03 2.5e-04 5.0e-04
 7.5e-04 2.5e-05 5.0e-05 7.5e-05]
Deviation tag: domega_dtau_scan_M1e6_q4_dense
Default domega_dict: {'2,2': 0.01, '2,1': 0.0, '3,3': 0.0, '4,4': 0.0, '4,3': 0.0, '5,5': 0.0}
Default dtau_dict: {'2,2': 0.01, '2,1': 0.0, '3,3': 0.0, '4,4': 0.0, '4,3': 0.0, '5,5': 0.0}


In [7]:
# ------------------------------------------------------------
# Setting the pSEOB configuration used for constructing the multiplier
# ------------------------------------------------------------

waveform_params_pseob_gr = {
    "t0": 0.0,
    "minf": 1e-5,
    "maxf": 0.2,

    "T_obs": 60 * 86400,
    "max_td_samples": 2_000_000,
    "deltaT": None,

    "approximant": "SEOBNRv5HM",
    "modes": common_modes,
    "n_freq": 10000,

    "timetomerger_max": 2.0,
    "DeltalnMf_max": 0.01,
    "TDIrescaled": True,
    "TDI": "TDIAET",
    "LISAnoise": copy.deepcopy(pyLISAnoise.LISAnoiseSciRDv1),
}

waveform_params_pseob_gr["LISAnoise"]["WDbackground"] = True
waveform_params_pseob_gr["LISAnoise"]["WDduration"] = 4.0

# Here we keep a default deviated configuration and rebuild the deviation dictionaries for every injection inside the main loop
waveform_params_pseob_dev = copy.deepcopy(waveform_params_pseob_gr)
waveform_params_pseob_dev.update({
    "domega_dict": domega_dict,
    "dtau_dict": dtau_dict,
})

# Reusing the GR configuration as the default in the pSEOB helper functions
waveform_params_base = waveform_params_pseob_gr

# ------------------------------------------------------------
# Setting the ROM carrier and LISA response configuration
# ------------------------------------------------------------

waveform_params_rom = {
    "minf": 1e-5,
    "maxf": 0.5,
    "t0": 0.0,
    "timetomerger_max": 1.0,
    "fend": None,
    "tmin": None,
    "tmax": None,
    "phiref": 0.0,
    "fref_for_phiref": 0.0,
    "tref": 0.0,
    "fref_for_tref": 0.0,
    "force_phiref_fref": True,
    "toffset": 0.0,
    "modes": common_modes,
    "TDI": "TDIAET",
    "acc": 0.0001,
    "order_fresnel_stencil": 0,
    "approximant": "SEOBNRv5HMROM",
    "LISAconst": "Proposal",
    "responseapprox": "full",
    "frozenLISA": False,
    "TDIrescaled": True,
    "LISAnoise": {
        "InstrumentalNoise": "SciRDv1",
        "WDbackground": True,
        "WDduration": 4.0,
        "lowf_add_pm_noise_f0": 0.0,
        "lowf_add_pm_noise_alpha": 2.0,
    },
}

print("pSEOB approximant used for multipliers:", waveform_params_pseob_gr["approximant"])
print("ROM carrier approximant:", waveform_params_rom["approximant"])


pSEOB approximant used for multipliers: SEOBNRv5HM
ROM carrier approximant: SEOBNRv5HMROM


## 4. Constructing the pSEOB waveforms

Generating the pSEOB modes in the time domain, applying the time-domain window, and transforming each mode onto a common frequency grid.


In [8]:
# Estimating time-domain downsampling from the spline interpolation error

def Deltat_insp(nu, t, acc=1e-4, tc=0.):
    Lambda = 8./3 * np.pi**(5./12) * (acc * nu)**(1./4)
    return 8*Lambda/3 * 5./(256*nu) * (256*nu/5 * (tc-t))**(27/32.)

def downsamplemask_arr_byfunc(x, func_dx):
    mask_ds = np.zeros_like(x, dtype=bool)
    i = 0
    while (i < len(x)-1):
        mask_ds[i] = True
        dx = x[i+1] - x[i]
        dx_func = func_dx(x[i])
        i += int(dx_func/dx)
    mask_ds[len(x)-1] = True
    return mask_ds


In [9]:
def mode_key_variants(lm):
    l, m = lm
    return [lm, (l, m), f"{l},{m}", f"{l}{m}", f"h{l}{m}"]


def get_mode_from_dict(hlm, lm):
    for key in mode_key_variants(lm):
        if key in hlm:
            return hlm[key]
    raise KeyError(f"Mode {lm} not found. Available keys: {list(hlm.keys())[:20]}")


def choose_f_start_from_Tobs(m1, m2, waveform_params):
    f_min_user = waveform_params.get("minf", 1e-5)
    T_obs = waveform_params.get("T_obs", 60 * 86400)
    f_from_T = pytools.funcNewtonianfoft(m1, m2, T_obs)
    return max(f_min_user, f_from_T), T_obs, f_from_T


def choose_deltaT(M, waveform_params):
    maxf = float(waveform_params.get("maxf", 0.2))
    user_deltaT = waveform_params.get("deltaT", None)

    M_seconds = M * pyconstants.MTSUN_SI
    f_rd_est = 0.08 / M_seconds

    delta_from_maxf = 1.0 / (4.0 * maxf)
    delta_from_ringdown = 1.0 / (8.0 * f_rd_est)
    auto_deltaT = min(delta_from_maxf, delta_from_ringdown)

    if user_deltaT is None:
        return auto_deltaT, f_rd_est

    return min(float(user_deltaT), auto_deltaT), f_rd_est


def generate_pseob_td(params, modes=None, waveform_params=None, verbose=True):

    if waveform_params is None:
        waveform_params = waveform_params_base

    if modes is None:
        modes = waveform_params.get(
            "modes",
            [(2,2), (2,1), (3,3), (4,4), (4,3), (5,5)]
        )

    M = float(params["M"])
    q = float(params["q"])

    m1 = M * q / (1.0 + q)
    m2 = M / (1.0 + q)

    chi1 = float(params.get("chi1", 0.0))
    chi2 = float(params.get("chi2", 0.0))

    distance = float(params.get("dist", 1.0))
    inclination = float(params.get("inc", 0.0))
    phi_ref = float(params.get("phi", 0.0))

    f_min, T_obs, f_from_T = choose_f_start_from_Tobs(m1, m2, waveform_params)
    f_max = float(waveform_params.get("maxf", 0.2))

    deltaT, f_rd_est = choose_deltaT(M, waveform_params)

    approx = waveform_params.get("approximant", "SEOBNRv5HM")

    params_pseob = {
        "mass1": m1,
        "mass2": m2,
        "spin1x": 0.0,
        "spin1y": 0.0,
        "spin1z": chi1,
        "spin2x": 0.0,
        "spin2y": 0.0,
        "spin2z": chi2,
        "distance": distance,
        "inclination": inclination,
        "phi_ref": phi_ref,
        "f22_start": f_min,
        "f_ref": f_min,
        "f_max": f_max,
        "deltaT": deltaT,
        "approximant": approx,
        "mode_array": modes,
    }

    for key in ["domega_dict", "dtau_dict", "dA_dict", "dw_dict", "lmax_nyquist"]:
        if key in waveform_params:
            params_pseob[key] = waveform_params[key]

    if verbose:
        print(
            f"pySEOBNR TD: M={M:.3e}, q={q:.3f}, "
            f"chi1={chi1:.3f}, chi2={chi2:.3f}, "
            f"f22_start={f_min:.3e} Hz, f_from_T={f_from_T:.3e} Hz, "
            f"f_RD_est={f_rd_est:.3e} Hz, f_max={f_max:.3e} Hz, "
            f"deltaT={deltaT:.3e} s"
        )

    times, hlm_raw = GenerateWaveform(params_pseob).generate_td_modes()
    times = np.asarray(times)

    hlm_out = {}

    for lm in modes:
        try:
            hlm_out[lm] = np.asarray(get_mode_from_dict(hlm_raw, lm), dtype=complex)
        except KeyError:
            warnings.warn(f"Requested mode {lm} was not returned by pySEOBNR. Skipping.")

    if not hlm_out:
        raise RuntimeError(f"No requested modes returned. Available keys: {list(hlm_raw.keys())[:20]}")

    return times, hlm_out


def generate_pseob_fft(
    params,
    modes=None,
    waveform_params=None,
    verbose=True,
):

    if waveform_params is None:
        waveform_params = waveform_params_base

    if modes is None:
        modes = waveform_params.get(
            "modes",
            [(2,2), (2,1), (3,3), (4,4), (4,3), (5,5)]
        )

    Deltat_w_end = 40.0
    Deltat_w_beg = 10000.0

    minf = float(waveform_params.get("minf", 1e-5))
    maxf = float(waveform_params.get("maxf", 0.2))
    n_freq = int(waveform_params.get("n_freq", 10000))

    # Generating the pSEOB modes in the time domain

    t, h = generate_pseob_td(
        params,
        modes=modes,
        waveform_params=waveform_params,
        verbose=verbose,
    )

    # Checking the uniform time sampling required by the FFT
    dt_arr = np.diff(t)

    if len(dt_arr) == 0:
        raise ValueError("The time-domain waveform has fewer than two samples.")

    if not np.allclose(dt_arr, dt_arr[0], rtol=1e-5, atol=0):
        raise ValueError(
            "pySEOBNR returned a non-uniform time grid. "
            "This original-style FFT cell assumes uniform sampling."
        )


    w = pytools.window_planck_vec(t, t[0], t[-1], Deltat_w_beg, Deltat_w_end)

    hlm_fft = {}

    for lm in modes:
        if lm not in h:
            warnings.warn(f"Mode {lm} missing from pSEOB TD output. Skipping.")
            continue

        hlm_ts = np.array([
            t,
            w * np.real(h[lm]),
            w * np.imag(h[lm])
        ]).T

        hlm_ts = pytools.zeropad(hlm_ts, extend=1)

        # Using the same positive-frequency FFT convention as the injection code
        hlm_fs = pytools.fft_positivef(hlm_ts)

        freq = hlm_fs[:, 0]
        hf = hlm_fs[:, 1] + 1j * hlm_fs[:, 2]

        amp = np.abs(hf)
        phase = np.unwrap(np.angle(hf))

        hlm_fft[lm] = {}
        hlm_fft[lm]["freq"] = freq
        hlm_fft[lm]["amp"] = amp
        hlm_fft[lm]["phase"] = phase

    if not hlm_fft:
        raise RuntimeError("No modes were converted to frequency domain.")


    # Interpolating the modes onto a common geometric frequency grid


    f_low_available = max([hlm_fft[lm]["freq"][0] for lm in hlm_fft])
    f_high_available = min([hlm_fft[lm]["freq"][-1] for lm in hlm_fft])

    if minf < f_low_available:
        raise ValueError(
            f"minf={minf:.6e} is not covered by all pSEOB FFT modes. "
            f"The common lower limit is {f_low_available:.6e}."
        )

    if maxf > f_high_available:
        raise ValueError(
            f"maxf={maxf:.6e} is not covered by all pSEOB FFT modes. "
            f"The common upper limit is {f_high_available:.6e}."
        )

    freq = np.geomspace(minf, maxf, n_freq)

    hlm_fd = {}

    for lm in modes:
        if lm not in hlm_fft:
            continue

        spline_amp_fft = pytools.spline(
            hlm_fft[lm]["freq"],
            hlm_fft[lm]["amp"]
        )

        spline_phase_fft = pytools.spline(
            hlm_fft[lm]["freq"],
            hlm_fft[lm]["phase"]
        )

        mask_fft = (
            (hlm_fft[lm]["freq"][0] <= freq)
            & (freq <= hlm_fft[lm]["freq"][-1])
        )

        amp = np.zeros_like(freq)
        phase = np.zeros_like(freq)

        amp[mask_fft] = spline_amp_fft(freq[mask_fft])
        phase[mask_fft] = spline_phase_fft(freq[mask_fft])

        hlm_fd[lm] = {}
        hlm_fd[lm]["freq"] = freq.copy()
        hlm_fd[lm]["amp"] = amp.copy()
        hlm_fd[lm]["phase"] = phase.copy()

        hlm_fd[lm]["tf"] = (
            1.0 / (2.0 * np.pi)
            * pytools.spline(
                hlm_fd[lm]["freq"],
                hlm_fd[lm]["phase"]
            )(hlm_fd[lm]["freq"], 1)
        )

        hlm_fd[lm]["h"] = (
            hlm_fd[lm]["amp"]
            * np.exp(1j * hlm_fd[lm]["phase"])
        )

    if verbose:
        print("pSEOB FD construction:")
        print("  modes kept =", list(hlm_fd.keys()))
        print("  freq range =", freq[0], freq[-1])
        print("  n_freq     =", len(freq))

    return hlm_fd


In [10]:
def build_wftdi_from_hlm_fd(
    params,
    hlm_fd,
    modes,
    t0=0.0,
    TDI="TDIAET",
    TDIrescaled=True,
):

    params_local = copy.deepcopy(params)

    if params_local.get("Lframe", False):
        params_local = lisatools.convert_Lframe_to_SSBframe(params_local, t0=t0)
        params_local["Lframe"] = False

    Deltat = params_local["Deltat"]
    inc = params_local["inc"]
    phi = params_local["phi"]
    lambd = params_local["lambda"]
    beta = params_local["beta"]
    psi = params_local["psi"]

    wftdi = {"modes": []}

    for lm in modes:
        if lm not in hlm_fd:
            warnings.warn(f"Mode {lm} missing from hlm_fd. Skipping.")
            continue

        freq = np.asarray(hlm_fd[lm]["freq"])
        amp = np.asarray(hlm_fd[lm]["amp"])
        phase = np.asarray(hlm_fd[lm]["phase"])
        tf = np.asarray(hlm_fd[lm]["tf"])

        good = (
            np.isfinite(freq)
            & np.isfinite(amp)
            & np.isfinite(phase)
            & np.isfinite(tf)
            & (freq > 0)
        )

        freq = freq[good]
        amp = amp[good]
        phase = phase[good]
        tf = tf[good]

        if len(freq) < 10:
            warnings.warn(f"Mode {lm} has too few valid FD samples. Skipping.")
            continue

        wftdi["modes"].append(lm)

        wftdi[lm] = {}
        wftdi[lm]["freq"] = freq
        wftdi[lm]["amp"] = amp
        wftdi[lm]["phase"] = phase + (2.0 * np.pi) * freq * Deltat
        wftdi[lm]["tf"] = tf + Deltat

        tdiClass = pyresponse.LISAFDresponseTDI3Chan(
            wftdi[lm]["freq"],
            wftdi[lm]["tf"],
            t0,
            lm[0],
            lm[1],
            inc,
            phi,
            lambd,
            beta,
            psi,
            TDI=TDI,
            TDIrescaled=TDIrescaled,
        )

        (
            wftdi[lm]["phaseRdelay"],
            wftdi[lm]["transferL1"],
            wftdi[lm]["transferL2"],
            wftdi[lm]["transferL3"],
        ) = tdiClass.get_response()

    if not wftdi["modes"]:
        raise RuntimeError("No modes were available for TDI response construction.")

    return params_local, wftdi


def make_frequency_grid_from_hlm_fd(
    params,
    hlm_fd,
    waveform_params=None,
    Deltalnf=5e-4,
    refine=1,
):

    if waveform_params is None:
        waveform_params = waveform_params_base

    M = float(params["M"])
    q = float(params["q"])

    m1 = M * q / (1.0 + q)
    m2 = M / (1.0 + q)

    modes = waveform_params.get("modes", list(hlm_fd.keys()))

    f_lows = []
    f_highs = []

    for lm in modes:
        if lm not in hlm_fd:
            continue

        f = np.asarray(hlm_fd[lm]["freq"])
        f = f[np.isfinite(f) & (f > 0)]

        if len(f) < 2:
            continue

        f_lows.append(f[0])
        f_highs.append(f[-1])

    if len(f_lows) == 0:
        raise RuntimeError("Cannot build frequency grid: no valid mode frequency arrays.")

    fLow = np.fmax(max(f_lows), waveform_params.get("minf", 1e-5))
    fHigh = np.fmin(min(f_highs), waveform_params.get("maxf", 0.2))

    T_est = pytools.funcNewtoniantoff(m1, m2, fLow)
    T = np.fmin(pyconstants.YRSID_SI, np.fmax(30.0 * 86400.0, T_est))

    fLow = np.fmax(fLow, pytools.funcNewtonianfoft(m1, m2, T))

    if not (fLow < fHigh):
        raise ValueError(f"Bad response frequency range: fLow={fLow}, fHigh={fHigh}")

    freq_nyq_log = pytools.composite_nyquist_log_freq_grid(
        fLow,
        fHigh,
        m1,
        m2,
        Deltalnf=Deltalnf,
        refine=refine,
    )

    return freq_nyq_log


## 5. Build the mode-by-mode multiplier

We compare the GR and deviated pSEOB modes on their common frequency support and constructing

$$
R_{\ell m}(f)=\frac{A_{\ell m}^{\mathrm{dev}}(f)}{A_{\ell m}^{\mathrm{GR}}(f)}
\exp\!\left[i\,\Delta\phi_{\ell m}(f)\right].
$$

Only a constant phase offset is removed at the lowest retained frequency. Any frequency-dependent phase difference is kept.


In [11]:
def interp_to_reference_frequency(mode_data, f_ref):

    f = np.asarray(mode_data["freq"])
    amp = np.asarray(mode_data["amp"])
    phase = np.unwrap(np.asarray(mode_data["phase"]))

    good = np.isfinite(f) & np.isfinite(amp) & np.isfinite(phase) & (f > 0) & (amp >= 0)
    f = f[good]
    amp = amp[good]
    phase = phase[good]

    order = np.argsort(f)
    f = f[order]
    amp = amp[order]
    phase = phase[order]

    mask = (f_ref >= f[0]) & (f_ref <= f[-1])
    amp_ref = np.full_like(f_ref, np.nan, dtype=float)
    phase_ref = np.full_like(f_ref, np.nan, dtype=float)

    amp_ref[mask] = pytools.spline(f, amp)(f_ref[mask])
    phase_ref[mask] = pytools.spline(f, phase)(f_ref[mask])

    return amp_ref, phase_ref, mask


def compute_mode_comparison(hlm_fd_gr, hlm_fd_dev, modes=None, align_phase_at_low_frequency=True):
    """
    Compute mode-by-mode comparison quantities.

    For each mode, returns:
        freq       : reference frequency grid
        amp_gr     : |h_lm^GR(f)|
        amp_dev    : |h_lm^dev(f)|
        amp_ratio  : |h_lm^dev(f)| / |h_lm^GR(f)|
        phase_gr   : unwrapped GR phase
        phase_dev  : unwrapped deviated phase
        dphase     : phase_dev - phase_gr, optionally shifted so dphase starts from zero
        h_gr, h_dev: reconstructed complex FD modes

    The optional low-frequency alignment removes a constant phase offset only.
    It does not remove any frequency-dependent physical phase shift.
    """
    if modes is None:
        modes = [lm for lm in hlm_fd_gr.keys() if lm in hlm_fd_dev]

    comparison = {}

    for lm in modes:
        if lm not in hlm_fd_gr or lm not in hlm_fd_dev:
            warnings.warn(f"Mode {lm} missing from one waveform. Skipping.")
            continue

        f_gr = np.asarray(hlm_fd_gr[lm]["freq"])
        f_dev = np.asarray(hlm_fd_dev[lm]["freq"])

        # Restricting the GR reference grid to the common frequency support
        f_low = max(np.nanmin(f_gr), np.nanmin(f_dev))
        f_high = min(np.nanmax(f_gr), np.nanmax(f_dev))
        f_ref = f_gr[(f_gr >= f_low) & (f_gr <= f_high)]

        if len(f_ref) < 10:
            warnings.warn(f"Mode {lm} has too few common frequency samples. Skipping.")
            continue

        amp_gr, phase_gr, mask_gr = interp_to_reference_frequency(hlm_fd_gr[lm], f_ref)
        amp_dev, phase_dev, mask_dev = interp_to_reference_frequency(hlm_fd_dev[lm], f_ref)

        amp_gr_max = np.nanmax(amp_gr)
        amp_floor = 1e-8 * amp_gr_max


        valid = mask_gr & mask_dev & np.isfinite(amp_gr) & np.isfinite(amp_dev) & np.isfinite(phase_gr) & np.isfinite(phase_dev) & (amp_gr > amp_floor)

        f_ref = f_ref[valid]
        amp_gr = amp_gr[valid]
        amp_dev = amp_dev[valid]
        phase_gr = np.unwrap(phase_gr[valid])
        phase_dev = np.unwrap(phase_dev[valid])

        amp_ratio = amp_dev / amp_gr
        dphase = np.unwrap(phase_dev - phase_gr)

        if align_phase_at_low_frequency and len(dphase) > 0:
            dphase = dphase - dphase[0]

        comparison[lm] = {
            "freq": f_ref,
            "amp_gr": amp_gr,
            "amp_dev": amp_dev,
            "amp_ratio": amp_ratio,
            "phase_gr": phase_gr,
            "phase_dev": phase_dev,
            "dphase": dphase,
            "h_gr": amp_gr * np.exp(1j * phase_gr),
            "h_dev": amp_dev * np.exp(1j * phase_dev),
        }

    return comparison


### Diagnostic plots

Plotting the pSEOB amplitudes, amplitude ratios, and phase differences mode by mode when a closer inspection is needed.


In [12]:
def plot_waveforms_mode_by_mode(mode_comparison, modes=None):
    """Plotting the GR and deviated amplitude and phase for each mode."""
    if modes is None:
        modes = list(mode_comparison.keys())

    for lm in modes:
        data = mode_comparison[lm]
        f = data["freq"]

        plt.figure(figsize=(8, 5))
        plt.loglog(f, data["amp_gr"], label="GR")
        plt.loglog(f, data["amp_dev"], linestyle="--", label="deviated")
        plt.xlabel(r"$f$ [Hz]")
        plt.ylabel(rf"$|h_{{{lm[0]}{lm[1]}}}(f)|$")
        plt.title(rf"Mode $({lm[0]},{lm[1]})$: amplitude")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(8, 5))
        plt.semilogx(f, data["phase_gr"], label="GR")
        plt.semilogx(f, data["phase_dev"], linestyle="--", label="deviated")
        plt.xlabel(r"$f$ [Hz]")
        plt.ylabel(rf"$\phi_{{{lm[0]}{lm[1]}}}(f)$ [rad]")
        plt.title(rf"Mode $({lm[0]},{lm[1]})$: phase")
        plt.legend()
        plt.tight_layout()
        plt.show()


def plot_amplitude_ratios(mode_comparison, modes=None):
    """Plotting the deviated-to-GR amplitude ratio for each mode."""
    if modes is None:
        modes = list(mode_comparison.keys())

    for lm in modes:
        data = mode_comparison[lm]
        plt.figure(figsize=(8, 5))
        plt.semilogx(data["freq"], data["amp_ratio"])
        plt.axhline(1.0, linestyle=":", linewidth=1)
        plt.xlabel(r"$f$ [Hz]")
        plt.ylabel(rf"$|h_{{{lm[0]}{lm[1]}}}^{{\rm dev}}|/|h_{{{lm[0]}{lm[1]}}}^{{\rm GR}}|$")
        plt.title(rf"Mode $({lm[0]},{lm[1]})$: amplitude ratio")
        plt.tight_layout()
        plt.show()


def plot_phase_differences(mode_comparison, modes=None):
    """Plotting the deviated-minus-GR phase difference for each mode."""
    if modes is None:
        modes = list(mode_comparison.keys())

    for lm in modes:
        data = mode_comparison[lm]
        plt.figure(figsize=(8, 5))
        plt.semilogx(data["freq"], data["dphase"])
        plt.axhline(0.0, linestyle=":", linewidth=1)
        plt.xlabel(r"$f$ [Hz]")
        plt.ylabel(rf"$\Delta \phi_{{{lm[0]}{lm[1]}}}(f)$ [rad]")
        plt.title(rf"Mode $({lm[0]},{lm[1]})$: phase difference, deviated $-$ GR")
        plt.tight_layout()
        plt.show()


def plot_all_comparisons(mode_comparison, modes=None):
    """Running all three diagnostic plots mode by mode."""
    plot_waveforms_mode_by_mode(mode_comparison, modes=modes)
    plot_amplitude_ratios(mode_comparison, modes=modes)
    plot_phase_differences(mode_comparison, modes=modes)


## 6. Injection parameters and output helpers

Loading one scan row at a time, applying its deviation to the ROM carrier, and writing both the summed and mode-resolved TDI data.


In [13]:
def validate_scan_params_file(params_file, mass_value):
    """Checking that an existing parameter file matches the requested mass scan."""
    with h5py.File(params_file, "r") as hf:
        if "M" not in hf:
            raise KeyError(f"Dataset 'M' is missing from {params_file}")

        if len(hf["M"]) != N:
            raise ValueError(
                f"{params_file} contains {len(hf['M'])} rows, but {N} are required."
            )

        stored_masses = np.asarray(hf["M"][:N], dtype=float)
        if not np.allclose(stored_masses, float(mass_value)):
            raise ValueError(
                f"Mass mismatch in {params_file}: expected {mass_value:.6e} Msun, "
                f"found {stored_masses}."
            )

        for key, expected in (("q", np.full(N, MASS_RATIO)),
                              ("d_omega", deviation_values),
                              ("d_tau", deviation_values)):
            if key not in hf or hf[key].shape != (N,):
                raise ValueError(f"Missing or incorrect {key} grid in {params_file}")
            if not np.allclose(hf[key][:], expected, rtol=1e-12, atol=0.0):
                raise ValueError(f"{key} does not match the dense scan in {params_file}")


def create_or_load_scan_params_file(mass_value, mass_label):
    """Loading or creating one 18-row parameter file for a given mass."""
    filename = (
        "params_injection_set_" + deviation_tag + "_" + mass_label + ".h5"
    )

    output_params_file = os.path.join(Output_root, filename)
    input_params_file = os.path.join(Input_scan_dir, filename)

    # Reusing the local output parameter file before checking the optional input folder
    for candidate in (output_params_file, input_params_file):
        if os.path.exists(candidate):
            validate_scan_params_file(candidate, mass_value)
            print("Using existing scan parameter file:", candidate)
            return candidate

    os.makedirs(Output_root, exist_ok=True)
    params_file = output_params_file

    print(f"Creating the fixed {mass_label} scan parameter file:", params_file)

    q = MASS_RATIO * np.ones(N, dtype=float)
    chi1 = 0.5 * np.ones(N, dtype=float)
    chi2 = 0.5 * np.ones(N, dtype=float)
    d_omega_arr = deviation_values.copy()
    d_tau_arr = deviation_values.copy()

    dist_base = cosmo.luminosity_distance(1.0).to_value(u.Mpc)
    dist = dist_base * np.ones(N, dtype=float)
    Deltat = np.zeros(N, dtype=float)
    inc = (np.pi / 3) * np.ones(N, dtype=float)
    phi = 0.7 * np.ones(N, dtype=float)
    lambda_ = 1.0 * np.ones(N, dtype=float)
    beta = (np.pi / 6) * np.ones(N, dtype=float)
    psi = 1.2 * np.ones(N, dtype=float)
    Lframe = True

    with h5py.File(params_file, "w") as hf:
        hf.create_dataset("M", data=float(mass_value) * np.ones(N, dtype=float))
        hf.create_dataset("q", data=q)
        hf.create_dataset("chi1", data=chi1)
        hf.create_dataset("chi2", data=chi2)
        hf.create_dataset("d_omega", data=d_omega_arr)
        hf.create_dataset("d_tau", data=d_tau_arr)
        hf.create_dataset("Deltat", data=Deltat)
        hf.create_dataset("dist", data=dist)
        hf.create_dataset("inc", data=inc)
        hf.create_dataset("phi", data=phi)
        hf.create_dataset("lambda", data=lambda_)
        hf.create_dataset("beta", data=beta)
        hf.create_dataset("psi", data=psi)
        hf.create_dataset("Lframe", data=np.array([int(Lframe)]))
        hf.attrs["mass_label"] = mass_label
        hf.attrs["mass_value"] = float(mass_value)
        hf.attrs["deviation_tag"] = deviation_tag
        hf.attrs["deviation_values"] = json.dumps(deviation_values.tolist())
        hf.attrs["deviation_modes"] = json.dumps(
            [list(lm) for lm in deviation_modes]
        )

    return params_file


def load_injection_params(params_file, inj_id=0):
    list_params = [
        "M", "q", "chi1", "chi2", "d_omega", "d_tau",
        "Deltat", "dist", "inc", "phi", "lambda", "beta", "psi",
    ]

    params = {}
    with h5py.File(params_file, "r") as hf:
        for param in list_params:
            if param in hf:
                params[param] = float(hf[param][inj_id])
        params["Lframe"] = bool(hf["Lframe"][:][0])

    # Recovering d_omega and d_tau from the scan array when loading older files
    params.setdefault("d_omega", float(deviation_values[inj_id]))
    params.setdefault("d_tau", float(deviation_values[inj_id]))

    return params


def safe_write_check(*paths, overwrite=False):
    """Checking output paths before allowing files to be overwritten."""
    if overwrite:
        return

    existing = [path for path in paths if os.path.exists(path)]
    if existing:
        message = "Refusing to overwrite existing files:\n" + "\n".join(existing)
        message += (
            "\nSet OVERWRITE_OUTPUTS = True only if you intentionally want "
            "to replace them."
        )
        raise FileExistsError(message)


In [14]:
def build_pseob_multiplier(params, modes, verbose=True):
    """
    Generate pSEOB GR and pSEOB-deviated FD modes and compute the mode-by-mode multiplier:
        amp_ratio(f) * exp(i dphase(f)).

    The dOmega/dTau dictionaries are rebuilt from params['d_omega'] and params['d_tau']
    for each inj_id, so this function can be used inside the scan loop.
    """
    d_omega_this = float(params.get("d_omega", 0.0))
    d_tau_this = float(params.get("d_tau", 0.0))
    domega_this, dtau_this = make_deviation_dicts(d_omega_this, d_tau_this, modes=modes)

    waveform_params_pseob_dev_this = copy.deepcopy(waveform_params_pseob_gr)
    waveform_params_pseob_dev_this.update({
        "domega_dict": domega_this,
        "dtau_dict": dtau_this,
    })

    if verbose:
        print("Generating pSEOB GR modes for multiplier...")

    hlm_fd_gr = generate_pseob_fft(
        params,
        modes=modes,
        waveform_params=waveform_params_pseob_gr,
        verbose=verbose,
    )

    if verbose:
        print("Generating pSEOB deviated modes for multiplier...")
        print(f"dOmega = {d_omega_this:g}, dTau = {d_tau_this:g}")
        print("domega_dict:", domega_this)
        print("dtau_dict:", dtau_this)

    hlm_fd_dev = generate_pseob_fft(
        params,
        modes=modes,
        waveform_params=waveform_params_pseob_dev_this,
        verbose=verbose,
    )

    common_generated_modes = [lm for lm in modes if lm in hlm_fd_gr and lm in hlm_fd_dev]

    mode_comparison = compute_mode_comparison(
        hlm_fd_gr,
        hlm_fd_dev,
        modes=common_generated_modes,
        align_phase_at_low_frequency=True,
    )

    return mode_comparison


def interp_mode_multiplier(mode_comparison, lm, f_target):

    # Interpolating the amplitude ratio and phase difference onto the ROM grid

    f_target = np.asarray(f_target)

    if lm not in mode_comparison:
        warnings.warn(f"No pSEOB multiplier found for mode {lm}. Leaving this ROM mode unchanged.")
        return np.ones_like(f_target, dtype=complex)

    data = mode_comparison[lm]
    f = np.asarray(data["freq"])
    amp_ratio = np.asarray(data["amp_ratio"])
    dphase = np.asarray(data["dphase"])

    good = (
        np.isfinite(f)
        & np.isfinite(amp_ratio)
        & np.isfinite(dphase)
        & (f > 0)
        & (amp_ratio > 0)
    )

    f = f[good]
    amp_ratio = amp_ratio[good]
    dphase = dphase[good]

    order = np.argsort(f)
    f = f[order]
    amp_ratio = amp_ratio[order]
    dphase = dphase[order]

    # Keeping the ROM mode unchanged outside the common frequency support.

    multiplier = np.ones_like(f_target, dtype=complex)
    mask = (f_target >= f[0]) & (f_target <= f[-1])

    amp_spline = pytools.spline(f, amp_ratio)
    phase_spline = pytools.spline(f, dphase)

    multiplier[mask] = amp_spline(f_target[mask]) * np.exp(1j * phase_spline(f_target[mask]))

    return multiplier


def generate_rom_tdi_modes(params, freq_grid, waveform_params=None):

    if waveform_params is None:
        waveform_params = waveform_params_rom

    # Removing the pSEOB-only deviation parameters before generating the ROM carrier
    params_rom = {k: v for k, v in params.items() if k not in ["d_omega", "d_tau"]}

    tdifreqseries_rom = lisa.GenerateLISATDIFreqseries_SMBH(
        params_rom,
        freq_grid,
        **waveform_params,
    )

    return tdifreqseries_rom


def apply_multiplier_to_rom_tdi(tdifreqseries_rom, mode_comparison, freq_grid, modes):
    """
    For each mode:
        TDI_ROM_modified_lm(f) = TDI_ROM_lm(f) * amp_ratio_lm(f) * exp(i dphase_lm(f)).

    Since the multiplier is a scalar function of frequency for a fixed mode, we apply the
    same multiplier to A, E, and T channel mode contributions.
    """
    modified_tdi = {
        "freq": freq_grid.copy(),
        "TDIA": np.zeros_like(freq_grid, dtype=complex),
        "TDIE": np.zeros_like(freq_grid, dtype=complex),
    }

    modified_modes = {}

    for lm in modes:
        if lm not in tdifreqseries_rom:
            warnings.warn(f"ROM generator did not return mode {lm}. Skipping.")
            continue

        mult = interp_mode_multiplier(mode_comparison, lm, freq_grid)

        chan1 = tdifreqseries_rom[lm]["chan1"] * mult
        chan2 = tdifreqseries_rom[lm]["chan2"] * mult

        modified_modes[lm] = {
            "freq": freq_grid.copy(),
            "TDIA": chan1,
            "TDIE": chan2,
            "multiplier": mult,
            "amp_multiplier": np.abs(mult),
            "phase_multiplier": np.unwrap(np.angle(mult)),
        }

        modified_tdi["TDIA"] += chan1
        modified_tdi["TDIE"] += chan2

    return modified_tdi, modified_modes


def save_modified_rom_outputs(out_dir, m_str, inj_id, modified_tdi, modified_modes, mode_comparison, params, overwrite=False):
    """
    Saving files with the same structure as the pSEOB injection-set files:
      1. data_*.h5   : summed TDI A/E data
      2. datalm_*.h5 : mode-by-mode TDI A/E contributions

    Also saving a small multiplier file to inspect exactly what was applied.
    """
    os.makedirs(out_dir, exist_ok=True)

    d_omega_this = float(params.get("d_omega", 0.0))
    d_tau_this = float(params.get("d_tau", 0.0))
    domega_this, dtau_this = make_deviation_dicts(d_omega_this, d_tau_this, modes=common_modes)

    data_out_file = os.path.join(out_dir, f"data_{output_tag}_{m_str}_id_{inj_id}.h5")
    datalm_out_file = os.path.join(out_dir, f"datalm_{output_tag}_{m_str}_id_{inj_id}.h5")
    multiplier_out_file = os.path.join(out_dir, f"multiplier_pseob_ratio_phase_{deviation_tag}_{m_str}_id_{inj_id}.h5")

    safe_write_check(data_out_file, datalm_out_file, multiplier_out_file, overwrite=overwrite)

    with h5py.File(data_out_file, "w") as hf:
        hf.create_dataset("freq", data=modified_tdi["freq"])
        hf.create_dataset("TDIA", data=modified_tdi["TDIA"])
        hf.create_dataset("TDIE", data=modified_tdi["TDIE"])

        hf.attrs["carrier_approximant"] = waveform_params_rom["approximant"]
        hf.attrs["multiplier_source_approximant"] = waveform_params_pseob_gr["approximant"]
        hf.attrs["deviation_tag"] = deviation_tag
        hf.attrs["mass_label"] = m_str
        hf.attrs["M"] = float(params["M"])
        hf.attrs["d_omega"] = d_omega_this
        hf.attrs["d_tau"] = d_tau_this
        hf.attrs["domega_dict"] = json.dumps(domega_this)
        hf.attrs["dtau_dict"] = json.dumps(dtau_this)
        hf.attrs["construction"] = "ROM mode TDI contribution multiplied by pSEOB amp_ratio*exp(i*dphase), then summed."

    with h5py.File(datalm_out_file, "w") as hf:
        for lm, data in modified_modes.items():
            g = hf.create_group(f"h{lm[0]}{lm[1]}")
            g.create_dataset("freq", data=data["freq"])
            g.create_dataset("TDIA", data=data["TDIA"])
            g.create_dataset("TDIE", data=data["TDIE"])

        hf.attrs["carrier_approximant"] = waveform_params_rom["approximant"]
        hf.attrs["multiplier_source_approximant"] = waveform_params_pseob_gr["approximant"]
        hf.attrs["deviation_tag"] = deviation_tag
        hf.attrs["mass_label"] = m_str
        hf.attrs["M"] = float(params["M"])
        hf.attrs["d_omega"] = d_omega_this
        hf.attrs["d_tau"] = d_tau_this
        hf.attrs["domega_dict"] = json.dumps(domega_this)
        hf.attrs["dtau_dict"] = json.dumps(dtau_this)

    with h5py.File(multiplier_out_file, "w") as hf:
        hf.attrs["deviation_tag"] = deviation_tag
        hf.attrs["mass_label"] = m_str
        hf.attrs["M"] = float(params["M"])
        hf.attrs["d_omega"] = d_omega_this
        hf.attrs["d_tau"] = d_tau_this
        hf.attrs["domega_dict"] = json.dumps(domega_this)
        hf.attrs["dtau_dict"] = json.dumps(dtau_this)
        hf.attrs["deviation_modes"] = json.dumps(deviation_modes)
        hf.attrs["params"] = json.dumps({
            k: float(v) if isinstance(v, (float, int, np.floating, np.integer, bool)) else str(v)
            for k, v in params.items()
        })

        for lm, data in mode_comparison.items():
            g = hf.create_group(f"h{lm[0]}{lm[1]}")
            g.create_dataset("freq", data=data["freq"])
            g.create_dataset("amp_ratio", data=data["amp_ratio"])
            g.create_dataset("dphase", data=data["dphase"])

    return data_out_file, datalm_out_file, multiplier_out_file


## 7. Generating the single-mass deviation scan

For each mass and injection ID, the calculation proceeds by

1. loading the mass-specific binary parameters and the corresponding $\delta\omega$, $\delta\tau$ values;
2. generating the GR and deviated pSEOB modes;
3. constructing the amplitude-ratio and phase-difference multiplier;
4. generating the ordinary `SEOBNRv5HMROM` TDI modes on the common frequency support;
5. applying the multiplier to each ROM mode and saving the summed and mode-resolved results.

The output tree contains only the `M1e6` subfolder. Every filename also contains its mass label and injection ID. A JSON manifest is written at the end to collect all paths needed by the recovery calculation.


In [15]:
# Let's run the full scan over the masses and injection IDs

generated_files = []
generation_records = []

for mass_value, mass_label in zip(MASS_VALUES, MASS_LABELS):
    params_file = create_or_load_scan_params_file(
        mass_value=mass_value,
        mass_label=mass_label,
    )

    out_dir = os.path.join(Output_root, mass_label)
    os.makedirs(out_dir, exist_ok=True)

    for inj_id in tqdm(
        range(N),
        desc=f"Generating {mass_label} dOmega/dTau scan",
    ):
        print("\n" + "=" * 80)
        print(f"Mass folder: {mass_label}, injection id: {inj_id}")
        print("Reading:", params_file)

        params = load_injection_params(params_file, inj_id=inj_id)
        d_omega_this = float(params["d_omega"])
        d_tau_this = float(params["d_tau"])
        print(f"Total mass: {params['M']:.6e} Msun")
        print(f"Scan value: dOmega = {d_omega_this:g}, dTau = {d_tau_this:g}")

        # Constructing the pSEOB multiplier for the current mass and scan point
        mode_comparison = build_pseob_multiplier(
            params,
            modes=common_modes,
            verbose=True,
        )

        # Using the common pSEOB support for defining the ROM response grid
        f_low = max(
            np.min(mode_comparison[lm]["freq"])
            for lm in mode_comparison
        )
        f_high = min(
            np.max(mode_comparison[lm]["freq"])
            for lm in mode_comparison
        )

        M = float(params["M"])
        q = float(params["q"])
        m1 = M * q / (1.0 + q)
        m2 = M / (1.0 + q)

        T_est = pytools.funcNewtoniantoff(m1, m2, f_low)
        T = np.fmin(
            pyconstants.YRSID_SI,
            np.fmax(30.0 * 86400.0, T_est),
        )
        f_low = np.fmax(f_low, pytools.funcNewtonianfoft(m1, m2, T))

        if not (f_low < f_high):
            raise ValueError(
                f"Bad ROM response frequency range for {mass_label}: "
                f"f_low={f_low}, f_high={f_high}"
            )

        freq_grid = pytools.composite_nyquist_log_freq_grid(
            f_low,
            f_high,
            m1,
            m2,
            Deltalnf=5e-4,
            refine=1,
        )

        print(
            f"ROM response grid: {len(freq_grid)} points, "
            f"{freq_grid[0]:.6e} -> {freq_grid[-1]:.6e} Hz"
        )

        # Generating the ordinary ROM TDI modes and applying each multiplier
        tdifreqseries_rom = generate_rom_tdi_modes(
            params,
            freq_grid,
            waveform_params=waveform_params_rom,
        )

        modified_tdi, modified_modes = apply_multiplier_to_rom_tdi(
            tdifreqseries_rom,
            mode_comparison,
            freq_grid,
            modes=common_modes,
        )

        data_out_file, datalm_out_file, multiplier_out_file = (
            save_modified_rom_outputs(
                out_dir,
                mass_label,
                inj_id,
                modified_tdi,
                modified_modes,
                mode_comparison,
                params,
                overwrite=OVERWRITE_OUTPUTS,
            )
        )

        generated_files.extend(
            [data_out_file, datalm_out_file, multiplier_out_file]
        )
        generation_records.append({
            "mass_label": mass_label,
            "M": M,
            "inj_id": inj_id,
            "d_omega": d_omega_this,
            "d_tau": d_tau_this,
            "params_file": params_file,
            "data_file": data_out_file,
            "datalm_file": datalm_out_file,
            "multiplier_file": multiplier_out_file,
        })

        print("Saved:", data_out_file)
        print("Saved:", datalm_out_file)
        print("Saved:", multiplier_out_file)

manifest_file = os.path.join(
    Output_root,
    f"generation_manifest_{deviation_tag}.json",
)
with open(manifest_file, "w", encoding="utf-8") as stream:
    json.dump(generation_records, stream, indent=2)

print(f"\nGenerated {len(generated_files)} HDF5 files.")
print("Saved manifest:", manifest_file)


ValueError: /pbs/home/a/amahdi/pySEOBNR_deviation_2/Data/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e6_q4_dense/params_injection_set_domega_dtau_scan_M1e6_q4_dense_M1e6.h5 contains 18 rows, but 13 are required.

## 8. Inspect one injection for each mass

Reading the same injection ID from the selected mass folder and checking the total TDI amplitudes, the individual mode contributions, and the multiplier itself.


In [ ]:
# Let's inspect one saved scan point for every mass

inj_id_inspect = 0
value = deviation_values[inj_id_inspect]

for mass_value, mass_label in zip(MASS_VALUES, MASS_LABELS):
    print("\n" + "=" * 80)
    print(
        f"Inspecting {mass_label}, id={inj_id_inspect}: "
        f"dOmega=dTau={value:g}"
    )

    data_file = os.path.join(
        Output_root,
        mass_label,
        f"data_{output_tag}_{mass_label}_id_{inj_id_inspect}.h5",
    )
    datalm_file = os.path.join(
        Output_root,
        mass_label,
        f"datalm_{output_tag}_{mass_label}_id_{inj_id_inspect}.h5",
    )
    multiplier_file = os.path.join(
        Output_root,
        mass_label,
        f"multiplier_pseob_ratio_phase_{deviation_tag}_{mass_label}_id_{inj_id_inspect}.h5",
    )

    print("Reading:", data_file)
    with h5py.File(data_file, "r") as hf:
        data = {key: hf[key][:] for key in hf.keys()}
        print(
            "d_omega:", hf.attrs.get("d_omega"),
            "d_tau:", hf.attrs.get("d_tau"),
        )

    plt.figure(figsize=(10, 5))
    plt.loglog(data["freq"], np.abs(data["TDIA"]), label="modified ROM TDIA")
    plt.loglog(data["freq"], np.abs(data["TDIE"]), label="modified ROM TDIE")
    plt.xlabel("f [Hz]")
    plt.ylabel("TDI amplitude")
    plt.title(
        fr"{mass_label}: modified ROM injection, "
        fr"$\delta\omega=\delta\tau={value:g}$"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("Reading:", datalm_file)
    with h5py.File(datalm_file, "r") as hf:
        plt.figure(figsize=(10, 5))
        for group in hf.keys():
            freq = hf[group]["freq"][:]
            channel_a = hf[group]["TDIA"][:]
            plt.loglog(freq, np.abs(channel_a), label=group)
        plt.xlabel("f [Hz]")
        plt.ylabel("|TDIA mode contribution|")
        plt.title(f"{mass_label}: mode contributions")
        plt.legend(ncol=2)
        plt.tight_layout()
        plt.show()

    print("Reading:", multiplier_file)
    with h5py.File(multiplier_file, "r") as hf:
        plt.figure(figsize=(10, 5))
        for group in hf.keys():
            freq = hf[group]["freq"][:]
            ratio = hf[group]["amp_ratio"][:]
            plt.semilogx(freq, ratio, label=group)
        plt.axhline(1.0, linestyle=":", linewidth=1)
        plt.xlabel("f [Hz]")
        plt.ylabel("pSEOB amplitude ratio")
        plt.title(f"{mass_label}: pSEOB multiplier")
        plt.legend(ncol=2)
        plt.tight_layout()
        plt.show()


## 9. Compare the full scan for each mass

Overlaying the total $A$-channel amplitude for every completed deviation point, using a separate figure for each mass.


In [ ]:
for mass_value, mass_label in zip(MASS_VALUES, MASS_LABELS):
    plt.figure(figsize=(10, 5))

    for inj_id, value in enumerate(deviation_values):
        data_file = os.path.join(
            Output_root,
            mass_label,
            f"data_{output_tag}_{mass_label}_id_{inj_id}.h5",
        )
        if not os.path.exists(data_file):
            print("Missing:", data_file)
            continue

        with h5py.File(data_file, "r") as hf:
            freq = hf["freq"][:]
            channel_a = hf["TDIA"][:]
            value_file = hf.attrs.get("d_omega", value)

        plt.loglog(
            freq,
            np.abs(channel_a),
            label=fr"$\delta\omega=\delta\tau={value_file:g}$",
        )

    mass_exponent = int(np.log10(mass_value))
    plt.xlabel("f [Hz]")
    plt.ylabel("Modified ROM TDIA amplitude")
    plt.title(
        fr"ROM $\times$ pSEOB ratio/phase scan, "
        fr"$M=10^{{{mass_exponent}}}M_\odot$"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()


## 10. Consistency checks for the selected mass

Reconstructing the ordinary ROM waveform for the zero-deviation injection of each mass and comparing it with the corresponding saved modified waveform. The multiplier and final waveform should reduce to their GR values within numerical tolerance.


In [ ]:
import pandas as pd


def prepare_consistency_case(mass_value, mass_label, inj_id=ZERO_DEVIATION_ID):
    """Loading one saved case and rebuilding its ordinary ROM waveform."""
    params_file = create_or_load_scan_params_file(
        mass_value=mass_value,
        mass_label=mass_label,
    )
    params_check = load_injection_params(params_file, inj_id=inj_id)

    out_dir = os.path.join(Output_root, mass_label)
    data_file = os.path.join(
        out_dir,
        f"data_{output_tag}_{mass_label}_id_{inj_id}.h5",
    )
    datalm_file = os.path.join(
        out_dir,
        f"datalm_{output_tag}_{mass_label}_id_{inj_id}.h5",
    )
    multiplier_file = os.path.join(
        out_dir,
        f"multiplier_pseob_ratio_phase_{deviation_tag}_{mass_label}_id_{inj_id}.h5",
    )

    for file_path in (data_file, datalm_file, multiplier_file):
        if not os.path.exists(file_path):
            raise FileNotFoundError(file_path)

    with h5py.File(data_file, "r") as hf:
        freq_grid = hf["freq"][:]
        modified_tdi = {
            "freq": freq_grid.copy(),
            "TDIA": hf["TDIA"][:],
            "TDIE": hf["TDIE"][:],
        }
        saved_mass = float(hf.attrs.get("M", params_check["M"]))
        saved_domega = float(
            hf.attrs.get("d_omega", params_check["d_omega"])
        )
        saved_dtau = float(
            hf.attrs.get("d_tau", params_check["d_tau"])
        )

    if not np.isclose(saved_mass, mass_value):
        raise ValueError(
            f"Mass mismatch for {mass_label}: params={mass_value}, file={saved_mass}"
        )
    if not np.isclose(saved_domega, params_check["d_omega"]):
        raise ValueError(
            f"d_omega mismatch for {mass_label}, inj_id={inj_id}: "
            f"params={params_check['d_omega']}, file={saved_domega}"
        )
    if not np.isclose(saved_dtau, params_check["d_tau"]):
        raise ValueError(
            f"d_tau mismatch for {mass_label}, inj_id={inj_id}: "
            f"params={params_check['d_tau']}, file={saved_dtau}"
        )

    modified_modes = {}
    with h5py.File(datalm_file, "r") as hf:
        for lm in common_modes:
            group = f"h{lm[0]}{lm[1]}"
            if group not in hf:
                continue
            modified_modes[lm] = {
                "freq": hf[group]["freq"][:],
                "TDIA": hf[group]["TDIA"][:],
                "TDIE": hf[group]["TDIE"][:],
            }

    tdifreqseries_rom = generate_rom_tdi_modes(
        params_check,
        freq_grid,
        waveform_params=waveform_params_rom,
    )

    rom_tdi = {
        "freq": freq_grid.copy(),
        "TDIA": np.zeros_like(freq_grid, dtype=complex),
        "TDIE": np.zeros_like(freq_grid, dtype=complex),
    }
    rom_modes = {}

    for lm in common_modes:
        if lm not in tdifreqseries_rom:
            continue

        rom_a = np.asarray(tdifreqseries_rom[lm]["chan1"])
        rom_e = np.asarray(tdifreqseries_rom[lm]["chan2"])

        if rom_a.shape != freq_grid.shape:
            raise ValueError(
                f"TDIA shape mismatch for {mass_label}, mode {lm}: "
                f"{rom_a.shape} != {freq_grid.shape}"
            )
        if rom_e.shape != freq_grid.shape:
            raise ValueError(
                f"TDIE shape mismatch for {mass_label}, mode {lm}: "
                f"{rom_e.shape} != {freq_grid.shape}"
            )

        rom_modes[lm] = {
            "freq": freq_grid.copy(),
            "TDIA": rom_a,
            "TDIE": rom_e,
        }
        rom_tdi["TDIA"] += rom_a
        rom_tdi["TDIE"] += rom_e

    return {
        "mass_value": float(mass_value),
        "mass_label": mass_label,
        "inj_id": inj_id,
        "params": params_check,
        "multiplier_file": multiplier_file,
        "modified_tdi": modified_tdi,
        "modified_modes": modified_modes,
        "rom_tdi": rom_tdi,
        "rom_modes": rom_modes,
    }


inj_id_check = ZERO_DEVIATION_ID
consistency_cases = {
    mass_label: prepare_consistency_case(
        mass_value,
        mass_label,
        inj_id=inj_id_check,
    )
    for mass_value, mass_label in zip(MASS_VALUES, MASS_LABELS)
}

for mass_label, case in consistency_cases.items():
    print("\n" + "=" * 80)
    print("mass_label =", mass_label)
    print("M =", case["params"]["M"])
    print("inj_id =", case["inj_id"])
    print("d_omega =", case["params"]["d_omega"])
    print("d_tau =", case["params"]["d_tau"])
    print("ROM modes:", list(case["rom_modes"].keys()))
    print("Modified modes:", list(case["modified_modes"].keys()))


In [ ]:
for mass_label, case in consistency_cases.items():
    for channel in ("TDIA", "TDIE"):
        plt.figure(figsize=(10, 5))
        plt.loglog(
            case["rom_tdi"]["freq"],
            np.abs(case["rom_tdi"][channel]),
            label=f"{channel}: ordinary SEOBNRv5HMROM",
        )
        plt.loglog(
            case["modified_tdi"]["freq"],
            np.abs(case["modified_tdi"][channel]),
            "--",
            label=(
                f"{channel}: modified ROM, inj_id={case['inj_id']}, "
                f"dOmega=dTau={case['params']['d_omega']:g}"
            ),
        )
        plt.xlabel(r"$f$ [Hz]")
        plt.ylabel(r"$|\tilde{h}(f)|$")
        plt.title(
            f"{mass_label}: total TDI comparison, "
            f"inj_id={case['inj_id']}, {channel}"
        )
        plt.legend()
        plt.grid(True, which="both", alpha=0.3)
        plt.tight_layout()
        plt.show()


In [ ]:
for mass_label, case in consistency_cases.items():
    for lm in common_modes:
        if lm not in case["rom_modes"] or lm not in case["modified_modes"]:
            continue

        for channel in ("TDIA", "TDIE"):
            plt.figure(figsize=(10, 5))
            plt.loglog(
                case["rom_modes"][lm]["freq"],
                np.abs(case["rom_modes"][lm][channel]),
                label=f"ordinary ROM {channel}",
            )
            plt.loglog(
                case["modified_modes"][lm]["freq"],
                np.abs(case["modified_modes"][lm][channel]),
                "--",
                label=f"modified ROM {channel}, inj_id={case['inj_id']}",
            )
            plt.xlabel(r"$f$ [Hz]")
            plt.ylabel(rf"$|\tilde{{h}}_{{{lm[0]}{lm[1]}}}(f)|$")
            plt.title(
                rf"{mass_label}: mode $({lm[0]},{lm[1]})$, "
                rf"inj_id={case['inj_id']}, {channel}"
            )
            plt.legend()
            plt.grid(True, which="both", alpha=0.3)
            plt.tight_layout()
            plt.show()


In [ ]:
multiplier_rows = []

for mass_label, case in consistency_cases.items():
    params_check = case["params"]

    with h5py.File(case["multiplier_file"], "r") as hf:
        multiplier_domega = float(
            hf.attrs.get("d_omega", params_check["d_omega"])
        )
        multiplier_dtau = float(
            hf.attrs.get("d_tau", params_check["d_tau"])
        )

        if not np.isclose(multiplier_domega, params_check["d_omega"]):
            raise ValueError(
                f"Multiplier d_omega mismatch for {mass_label}, "
                f"inj_id={case['inj_id']}"
            )
        if not np.isclose(multiplier_dtau, params_check["d_tau"]):
            raise ValueError(
                f"Multiplier d_tau mismatch for {mass_label}, "
                f"inj_id={case['inj_id']}"
            )

        for group in hf.keys():
            freq = hf[group]["freq"][:]
            amp_ratio = hf[group]["amp_ratio"][:]
            dphase = hf[group]["dphase"][:]

            good = (
                np.isfinite(freq)
                & np.isfinite(amp_ratio)
                & np.isfinite(dphase)
                & (freq > 0)
            )
            if not np.any(good):
                continue

            freq = freq[good]
            amp_ratio = amp_ratio[good]
            dphase = dphase[good]
            multiplier = amp_ratio * np.exp(1j * dphase)

            multiplier_rows.append({
                "mass_label": mass_label,
                "M": case["mass_value"],
                "mode": group,
                "N": len(freq),
                "f_min": np.min(freq),
                "f_max": np.max(freq),
                "max_abs_amp_ratio_minus_1": np.max(
                    np.abs(amp_ratio - 1.0)
                ),
                "rms_amp_ratio_minus_1": np.sqrt(
                    np.mean(np.abs(amp_ratio - 1.0) ** 2)
                ),
                "max_abs_dphase": np.max(np.abs(dphase)),
                "rms_dphase": np.sqrt(np.mean(dphase ** 2)),
                "max_abs_multiplier_minus_1": np.max(
                    np.abs(multiplier - 1.0)
                ),
                "rms_multiplier_minus_1": np.sqrt(
                    np.mean(np.abs(multiplier - 1.0) ** 2)
                ),
            })

df_multiplier_check = pd.DataFrame(multiplier_rows)
display(df_multiplier_check)


In [ ]:
mode_rows = []

for mass_label, case in consistency_cases.items():
    for lm in common_modes:
        if lm not in case["rom_modes"] or lm not in case["modified_modes"]:
            continue

        group = f"h{lm[0]}{lm[1]}"

        for channel in ("TDIA", "TDIE"):
            rom_h = case["rom_modes"][lm][channel]
            modified_h = case["modified_modes"][lm][channel]

            if rom_h.shape != modified_h.shape:
                raise ValueError(
                    f"Shape mismatch for {mass_label}, inj_id={case['inj_id']}, "
                    f"mode={lm}, channel={channel}: "
                    f"{rom_h.shape} != {modified_h.shape}"
                )

            scale = np.max(np.abs(rom_h))
            abs_diff = np.abs(modified_h - rom_h)

            mode_rows.append({
                "mass_label": mass_label,
                "M": case["mass_value"],
                "mode": group,
                "channel": channel,
                "max_abs_modified_minus_ROM": np.max(abs_diff),
                "relative_to_max_ROM": (
                    np.max(abs_diff) / scale if scale > 0 else np.nan
                ),
                "rms_relative_to_max_ROM": (
                    np.sqrt(np.mean(abs_diff ** 2)) / scale
                    if scale > 0 else np.nan
                ),
                "max_ROM_amp": scale,
                "max_modified_amp": np.max(np.abs(modified_h)),
            })

df_mode_check = pd.DataFrame(mode_rows)
display(df_mode_check)


In [ ]:
total_rows = []

for mass_label, case in consistency_cases.items():
    for channel in ("TDIA", "TDIE"):
        rom_h = case["rom_tdi"][channel]
        modified_h = case["modified_tdi"][channel]

        if rom_h.shape != modified_h.shape:
            raise ValueError(
                f"Total shape mismatch for {mass_label}, "
                f"inj_id={case['inj_id']}, channel={channel}: "
                f"{rom_h.shape} != {modified_h.shape}"
            )

        scale = np.max(np.abs(rom_h))
        abs_diff = np.abs(modified_h - rom_h)

        total_rows.append({
            "mass_label": mass_label,
            "M": case["mass_value"],
            "channel": channel,
            "max_abs_saved_modified_minus_ROM": np.max(abs_diff),
            "relative_to_max_ROM": (
                np.max(abs_diff) / scale if scale > 0 else np.nan
            ),
            "rms_relative_to_max_ROM": (
                np.sqrt(np.mean(abs_diff ** 2)) / scale
                if scale > 0 else np.nan
            ),
            "max_ROM_amp": scale,
            "max_saved_modified_amp": np.max(np.abs(modified_h)),
        })

df_total_check = pd.DataFrame(total_rows)
display(df_total_check)


In [ ]:
for mass_label, case in consistency_cases.items():
    for channel in ("TDIA", "TDIE"):
        scale = np.max(np.abs(case["rom_tdi"][channel]))

        plt.figure(figsize=(10, 5))
        plt.semilogx(
            case["rom_tdi"]["freq"],
            np.abs(
                case["modified_tdi"][channel]
                - case["rom_tdi"][channel]
            ) / scale,
            label=f"{mass_label}, {channel}: inj_id={case['inj_id']}",
        )
        plt.xlabel(r"$f$ [Hz]")
        plt.ylabel(
            r"$|h_{\rm modified}-h_{\rm ROM}|/"
            r"\max|h_{\rm ROM}|$"
        )
        plt.title(
            f"{mass_label}: ROM residual, "
            f"inj_id={case['inj_id']}, {channel}"
        )
        plt.legend()
        plt.tight_layout()
        plt.show()


In [ ]:
tol = 1e-8
identity_rows = []

for mass_label, case in consistency_cases.items():
    multiplier_subset = df_multiplier_check[
        df_multiplier_check["mass_label"] == mass_label
    ]
    mode_subset = df_mode_check[
        df_mode_check["mass_label"] == mass_label
    ]
    total_subset = df_total_check[
        df_total_check["mass_label"] == mass_label
    ]

    max_multiplier_error = (
        multiplier_subset["max_abs_multiplier_minus_1"].max()
        if not multiplier_subset.empty else np.nan
    )
    max_mode_error = (
        mode_subset["relative_to_max_ROM"].max()
        if not mode_subset.empty else np.nan
    )
    max_total_error = (
        total_subset["relative_to_max_ROM"].max()
        if not total_subset.empty else np.nan
    )

    params_check = case["params"]
    zero_deviation = (
        np.isclose(params_check["d_omega"], 0.0)
        and np.isclose(params_check["d_tau"], 0.0)
    )
    passed = (
        zero_deviation
        and np.isfinite(max_multiplier_error)
        and np.isfinite(max_total_error)
        and max_multiplier_error < tol
        and max_total_error < tol
    )

    identity_rows.append({
        "mass_label": mass_label,
        "M": case["mass_value"],
        "inj_id": case["inj_id"],
        "d_omega": params_check["d_omega"],
        "d_tau": params_check["d_tau"],
        "max_abs_multiplier_minus_1": max_multiplier_error,
        "max_mode_relative_ROM_error": max_mode_error,
        "max_total_relative_ROM_error": max_total_error,
        "status": "PASS" if passed else "FAIL",
    })

df_identity_check = pd.DataFrame(identity_rows)
display(df_identity_check)
